# 🃏 The Mind — LLMs + RL con Unsloth en Kaggle

| GPU Kaggle | VRAM | Modelo auto-seleccionado |
|------------|------|-------------------------|
| T4 x1 | 16 GB | Qwen2.5-7B-Instruct |
| T4 x2 | 32 GB | Qwen2.5-14B-Instruct |
| P100 | 16 GB | Qwen2.5-7B-Instruct |

**TPU**: no compatible (requiere torch_xla). Usa siempre GPU.

In [1]:
# !rm -rf /root/.cache/huggingface/hub

In [2]:
import os
import warnings

warnings.filterwarnings("ignore", message=".*Both `max_new_tokens` and `max_length`.*")

SAFE_MODE = True
if SAFE_MODE:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

    # 2. Desactivar el descargador agresivo (evita que se quede al 60%)
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    
    # 3. Tus bloqueos anteriores (para evitar el error de los 120s)
    os.environ["UNSLOTH_DISABLE_STATS"] = "1"
    os.environ["UNSLOTH_DISABLE_TELEMETRY"] = "1"

In [3]:
# ── CELDA 1: Instalar Unsloth + dependencias ──────────────────────────────────
# Unsloth tiene instaladores específicos por versión de CUDA.
# Kaggle usa CUDA 12.x, así que usamos el instalador genérico.
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-500:])
    return result.returncode == 0

# Unsloth: instalador oficial para Kaggle
run('pip install -q unsloth')
run('pip install -q --no-deps unsloth_zoo')
# Resto de dependencias que Kaggle no trae
run('pip install -q peft>=0.10.0 bitsandbytes>=0.43.0')

print('Instalación completada ✓')

Instalación completada ✓


In [4]:
# ── CELDA 2: Clonar repo ──────────────────────────────────────────────────────
import os, sys

REPO_URL  = 'https://github.com/JesusCristoII/The-Mind-LLMs'
REPO_DIR  = '/kaggle/working/The-Mind-LLMs'
CKPT_DIR  = '/kaggle/working/checkpoints'   # aquí van los checkpoints
OUT_DIR   = '/kaggle/working'               # Kaggle expone este directorio

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

os.makedirs(CKPT_DIR, exist_ok=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f'Repo: {REPO_DIR}')
print(f'Checkpoints: {CKPT_DIR}')
print(f'Archivos: {os.listdir(REPO_DIR)}')

fatal: could not create leading directories of '/kaggle/working/The-Mind-LLMs': Permission denied


PermissionError: [Errno 13] Permission denied: '/kaggle'

In [5]:
# ── CELDA 3: Detección de hardware ────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No se detectó GPU. Kaggle > Settings > Accelerator > GPU T4 x2')

DEVICE   = 'cuda'
n_gpus   = torch.cuda.device_count()
vram_gb  = sum(torch.cuda.get_device_properties(i).total_memory
               for i in range(n_gpus)) / 1e9
gpu_name = torch.cuda.get_device_name(0)
compute  = torch.cuda.get_device_capability(0)

print(f'GPU: {gpu_name} x{n_gpus}')
print(f'VRAM total: {vram_gb:.1f} GB')
print(f'Compute: sm_{compute[0]}{compute[1]}')

# Modelo según VRAM
if vram_gb >= 28:
    MODEL_NAME = 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit'
elif vram_gb >= 12:
    MODEL_NAME = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
else:
    MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'

# Unsloth gestiona la cuantización internamente — no necesitamos USE_4BIT manual
# Flash Attention solo en Ampere+ (sm_80). T4=sm_75, P100=sm_60
USE_FLASH_ATTN = compute[0] >= 8

print(f'\nModelo: {MODEL_NAME}')
print(f'Flash Attention: {USE_FLASH_ATTN}')

GPU: NVIDIA GeForce RTX 3050 Laptop GPU x1
VRAM total: 4.3 GB
Compute: sm_86

Modelo: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
Flash Attention: True


In [6]:
# ── CELDA 4: Cargar modelo con Unsloth ────────────────────────────────────────
# Unsloth reduce el uso de VRAM ~2x respecto a PEFT estándar:
#   - Kernels Triton optimizados para el forward pass
#   - Gradient checkpointing inteligente
#   - No duplica pesos en memoria durante el training

from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024
LORA_R      = 16    # rango LoRA: más alto = más capacidad, más VRAM
NUM_PLAYERS = 4
SHARED_LORA = True  # un solo adaptador compartido entre los 4 agentes

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,          # 4-bit por defecto para caber en T4
    dtype=None,                 # Unsloth elige el dtype óptimo
    trust_remote_code=True,
)

# Aplicar LoRA con Unsloth (más eficiente que PEFT estándar)
base_model = FastLanguageModel.get_peft_model(
    base_model,
    r=LORA_R,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],  # más módulos = mejor
    lora_alpha=LORA_R * 2,
    lora_dropout=0,             # 0 obligatorio para compatibilidad con autograd
    bias='none',
    use_gradient_checkpointing='unsloth',  # ahorra ~30% VRAM extra
    random_state=42,
)

print(f'Modelo cargado con Unsloth ✓')
print(f'Parámetros totales:    {base_model.num_parameters():,}')
base_model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/jesus/miniconda3/envs/entorno_universidad_311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3050 Laptop GPU. Num GPUs = 1. Max memory: 4.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 434/434 [00:03<00:00, 138.99it/s]


unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Modelo cargado con Unsloth ✓
Parámetros totales:    3,115,872,256
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [7]:
# ── CELDA 5: Asegurar chat_template ───────────────────────────────────────────
QWEN_CHAT_TEMPLATE = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}<|im_start|>system\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'user' %}<|im_start|>user\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'assistant' %}<|im_start|>assistant\n{{ message['content'] }}<|im_end|>\n"
    "{% endif %}{% endfor %}"
    "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"
)

if getattr(tokenizer, 'chat_template', None) is None:
    tokenizer.chat_template = QWEN_CHAT_TEMPLATE
    print('Chat template asignado ✓')
else:
    print('Chat template ya presente ✓')

test = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': 'hola'}],
    tokenize=False, add_generation_prompt=True
)
print(f'Test: {test[:80]}...')

Chat template ya presente ✓
Test: <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful ass...


In [8]:
# ── CELDA 6: Crear agentes con el modelo Unsloth ──────────────────────────────
# Como Unsloth ya aplicó LoRA al modelo, los agentes lo usan directamente
# sin que agents.py vuelva a aplicar PEFT. Hacemos un wrapper manual.

from agents import TheMindAgent

# Unsloth pone el modelo en modo inferencia rápida por defecto.
# Para entrenar, necesitamos desactivar eso.
FastLanguageModel.for_training(base_model)

agents = [
    TheMindAgent(
        player_id=i,
        model=base_model,
        tokenizer=tokenizer,
        device=DEVICE,
        max_new_tokens=80,
    )
    for i in range(NUM_PLAYERS)
]

print(f'{NUM_PLAYERS} agentes creados (modelo Unsloth compartido) ✓')

4 agentes creados (modelo Unsloth compartido) ✓


In [9]:
# ── CELDA 7: Test rápido de generación ────────────────────────────────────────
from environment import TheMindEnv

env   = TheMindEnv(num_players=NUM_PLAYERS)
state = env.reset(level=1)

print('Manos:', state.hands)

# Cambiar a modo inferencia para el test
FastLanguageModel.for_inference(base_model)

obs      = env.get_observation(0)
decision = agents[0].generate_action(obs)
print(f'Agente 0 → act={decision["action"]} | msg="{decision["message"]}"')

# Volver a modo training
FastLanguageModel.for_training(base_model)

Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'top_p', 'temperature', 'do_sample', 'max_new_tokens', 'pad_token_id', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Manos: {0: [26], 1: [24], 2: [20], 3: [12]}


/home/jesus/miniconda3/envs/entorno_universidad_311/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/jesus/miniconda3/envs/entorno_universidad_311/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/jesus/miniconda3/envs/entorno_universidad_311/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_at

Agente 0 → act=play | msg=""


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:
# ── CELDA 8: SFT previo al RL ─────────────────────────────────────────────────
from sft_trainer import run_sft, verify_sft_quality

# Durante SFT el modelo debe estar en modo training
FastLanguageModel.for_training(base_model)

sft_results = run_sft(
    agents=agents,
    tokenizer=tokenizer,
    # dataset_path=f'{REPO_DIR}/sft_dataset.json',
    dataset_path=f'sft_dataset.json',
    epochs=3,
    batch_size=4,
    lr=2e-4,
    max_length=1024,
    save_dir=f'{CKPT_DIR}/sft',
    device=DEVICE,
    shared_lora=SHARED_LORA,
)
print(f'SFT completado. Loss final: {sft_results["final_loss"]:.4f}')

In [ ]:
# ── CELDA 9: Verificar SFT ────────────────────────────────────────────────────
# Para inferencia de verificación
FastLanguageModel.for_inference(base_model)

json_ok_rate = verify_sft_quality(agents, tokenizer, num_samples=5, device=DEVICE)

# Volver a training para el RL
FastLanguageModel.for_training(base_model)

if json_ok_rate >= 0.8:
    print(f'\n✓ SFT exitoso ({json_ok_rate:.0%}). Listo para RL.')
else:
    print(f'\n⚠ Solo {json_ok_rate:.0%} JSON válidos. Considera más epochs.')

Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



VERIFICACIÓN POST-SFT


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ejemplo 1 ✓
  Mano: [4] | Mesa: 0 | Vidas: 3
  Generado: {"msg": "soy de los primeros", "act": "play"}
  Parseado → act=play, msg='soy de los primeros'


Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ejemplo 2 ✓
  Mano: [45] | Mesa: 0 | Vidas: 3
  Generado: {"msg": "voy muy tarde", "act": "wait"}
  Parseado → act=wait, msg='voy muy tarde'


Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ejemplo 3 ✓
  Mano: [12] | Mesa: 10 | Vidas: 2
  Generado: {"msg": "yo también ahora", "act": "play"}
  Parseado → act=play, msg='yo también ahora'


Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Ejemplo 4 ✓
  Mano: [33] | Mesa: 10 | Vidas: 2
  Generado: {"msg": "yo mucho después", "act": "wait"}
  Parseado → act=wait, msg='yo mucho después'

Ejemplo 5 ✓
  Mano: [7] | Mesa: 5 | Vidas: 1
  Generado: {"msg": "voy ya", "act": "play"}
  Parseado → act=play, msg='voy ya'

Resultado: 5/5 respuestas con JSON válido

✓ SFT exitoso (100%). Listo para RL.


In [ ]:
# ── CELDA 10: Configurar trainer RL ───────────────────────────────────────────
from trainer import GRPOTrainer, TrainerConfig
from utils import TrainingMetrics, LanguageAnalyzer, setup_logging
import logging

setup_logging(level='WARNING')

config = TrainerConfig(
    num_episodes=100,
    num_levels=3,
    group_size=4,
    lr=5e-6,
    kl_coeff=0.01,
    entropy_bonus=0.01,
    warmup_episodes=20,
    max_turns_per_episode=150,
    messages_per_turn=True,
    device=DEVICE,
    accumulate_grad_steps=4,
    checkpoint_every=5,
)

metrics       = TrainingMetrics()
lang_analyzer = LanguageAnalyzer()

trainer = GRPOTrainer(agents=agents, env=env, config=config)
print('Trainer RL listo ✓')

Trainer RL listo ✓


In [ ]:
# ── CELDA 11: Entrenamiento RL ────────────────────────────────────────────────
# IMPORTANTE: el trainer llama a model.train() y model.eval() internamente.
# Con Unsloth esto puede causar problemas, así que parcheamos el trainer
# para que use FastLanguageModel.for_training/for_inference en su lugar.

import types

# Parchear compute_policy_loss para activar training mode con Unsloth
_original_compute = trainer.compute_policy_loss

def _patched_compute(self, agent, prompt, output, advantage):
    FastLanguageModel.for_training(agent.model)
    loss = _original_compute(agent, prompt, output, advantage)
    return loss

trainer.compute_policy_loss = types.MethodType(_patched_compute, trainer)

# Parchear generate_action para usar modo inferencia rápida de Unsloth
_orig_generate = agents[0].__class__.generate_action

def _patched_generate(self, obs):
    FastLanguageModel.for_inference(self.model)
    result = _orig_generate(self, obs)
    return result

for agent in agents:
    agent.generate_action = types.MethodType(_patched_generate, agent)

print('Patches Unsloth aplicados ✓')
print(f'Checkpoints → {CKPT_DIR}')
print('='*60)

# Monkey-patch save_checkpoint para que guarde en CKPT_DIR
import utils as utils_module
_orig_save = utils_module.save_checkpoint

def _patched_save(agents, metrics, episode, output_dir='checkpoints', optimizers=None):
    # Forzar siempre CKPT_DIR independientemente de lo que pase el trainer
    _orig_save(agents, metrics, episode, output_dir=CKPT_DIR, optimizers=optimizers)
    # Guardar métricas en OUT_DIR para que Kaggle las exponga
    metrics.plot(f'{OUT_DIR}/metrics_ep{episode}.png')
    print(f'  → Checkpoint ep {episode} en {CKPT_DIR}/episode_{episode}/')

utils_module.save_checkpoint = _patched_save

try:
    trainer.train(
        metrics=metrics,
        lang_analyzer=lang_analyzer,
        verbose=True,
    )
except KeyboardInterrupt:
    print('\nEntrenamiento interrumpido por el usuario.')
except Exception as e:
    print(f'\nError durante entrenamiento: {e}')
    import traceback; traceback.print_exc()
finally:
    # Guardar siempre aunque haya error o interrupción
    print('\nGuardando checkpoint final...')
    _orig_save(agents, metrics, trainer.episode_count, output_dir=CKPT_DIR)
    metrics.plot(f'{OUT_DIR}/metrics_final.png')
    lang_analyzer.save_log(f'{OUT_DIR}/language_log.json')
    print('Guardado ✓')

Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Patches Unsloth aplicados ✓
Checkpoints → /kaggle/working/checkpoints


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=80

Ep    0 | Nivel 1 | Win: 0.00 | Reward: -11.22 | Errores: 1.0


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

In [ ]:
# ── CELDA 12: Análisis y resultados ───────────────────────────────────────────
import os

metrics.print_summary()
lang_analyzer.print_report()

print('\nÚltimos 15 mensajes de los agentes:')
for msg in lang_analyzer.message_log[-15:]:
    print(f'  Ep {msg["episode"]:3d} | J{msg["player"]}: {msg["message"]}')

print('\nCheckpoints guardados:')
if os.path.exists(CKPT_DIR):
    for d in sorted(os.listdir(CKPT_DIR)):
        print(f'  {CKPT_DIR}/{d}')

In [ ]:
# ── CELDA 13: [OPCIONAL] Reanudar entrenamiento ───────────────────────────────
# Sube los checkpoints a Kaggle como Dataset y ajusta CHECKPOINT_DIR.

# CHECKPOINT_DIR = '/kaggle/input/the-mind-checkpoints'
# RESUME_FROM    = 200
#
# from utils import load_checkpoint, list_checkpoints
# import torch
#
# list_checkpoints(CHECKPOINT_DIR)
#
# optimizers = [torch.optim.AdamW(a.model.parameters(), lr=config.lr)
#               for a in agents]
# metrics = load_checkpoint(
#     agents, episode=RESUME_FROM,
#     output_dir=CHECKPOINT_DIR, optimizers=optimizers
# )
# trainer = GRPOTrainer(agents=agents, env=env, config=config,
#                       optimizers=optimizers)
# trainer.episode_count = RESUME_FROM
# trainer.train(metrics=metrics, lang_analyzer=lang_analyzer)